# Class 2: Simple Transformations in Python
---

**Objective:** Learn the most common data transformations used in data engineering pipelines — all in pure Python with pandas.

**What you'll learn:**
1. Creating sample datasets from scratch
2. Column operations — rename, add, drop, reorder
3. Data type casting
4. String transformations
5. Date and time transformations
6. Handling nulls and missing data
7. Filtering and conditional logic
8. Aggregations and grouping
9. Sorting and deduplication
10. Joins
11. Pivot and unpivot (melt)
12. Practice exercises

> **Why this matters:** Every ETL/ELT pipeline — whether you're writing it in ADF, Fabric notebooks, dbt or PySpark — boils down to these operations. Master them in pandas first; the patterns transfer directly.


## 0. Setup


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta




## 1. Creating Sample Datasets

In real pipelines you'd pull from a database or API. For this class we'll build
our own data so we can control exactly what transformations we need to practice.


### 1.1 Customers dimension


In [2]:
customers = pd.DataFrame({
    "customer_id":   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "first_name":    ["Alice", "Bob", "  Charlie", "Diana", "Eve", "Frank", None, "Hannah", "Ivan", "Jane"],
    "last_name":     ["Smith", "JONES", "brown", "Williams", "Davis", "Miller", "Wilson", "Moore", "Taylor", "Anderson"],
    "email":         ["alice@mail.com", "bob@mail.com", "charlie@mail.com", None, "eve@mail.com",
                      "frank@mail.com", "g.wilson@mail.com", "hannah@mail.com", "ivan@mail.com", "jane@mail.com"],
    "city":          ["Nairobi", "London", "nairobi", "New York", "London", "Nairobi", "Berlin", "London", None, "Nairobi"],
    "country":       ["Kenya", "UK", "Kenya", "USA", "UK", "Kenya", "Germany", "UK", None, "Kenya"],
    "signup_date":   ["2023-01-15", "2023-02-20", "2023-03-10", "2023-01-05", "2023-04-18",
                      "2023-05-22", "2023-06-30", "2023-07-14", "2023-08-01", "2023-09-09"],
    "is_active":     [1, 1, 0, 1, 1, 0, 1, 1, 1, 0],
    "credit_limit":  [5000, 7500, 3000, 10000, 6000, None, 8000, 4500, None, 5500]
})
customers


,customer_id,first_name,last_name,email,city,country,signup_date,is_active,credit_limit
0,101,Alice,Smith,alice@mail.com,Nairobi,Kenya,2023-01-15,1,5000.0
1,102,Bob,JONES,bob@mail.com,London,UK,2023-02-20,1,7500.0
2,103,Charlie,brown,charlie@mail.com,nairobi,Kenya,2023-03-10,0,3000.0
3,104,Diana,Williams,NaN,New York,USA,2023-01-05,1,10000.0
4,105,Eve,Davis,eve@mail.com,London,UK,2023-04-18,1,6000.0
5,106,Frank,Miller,frank@mail.com,Nairobi,Kenya,2023-05-22,0,NaN
6,107,NaN,Wilson,g.wilson@mail.com,Berlin,Germany,2023-06-30,1,8000.0
7,108,Hannah,Moore,hannah@mail.com,London,UK,2023-07-14,1,4500.0
8,109,Ivan,Taylor,ivan@mail.com,NaN,NaN,2023-08-01,1,NaN
9,110,Jane,Anderson,jane@mail.com,Nairobi,Kenya,2023-09-09,0,5500.0


### 1.2 Products dimension


In [3]:
products = pd.DataFrame({
    "product_id":    ["P001", "P002", "P003", "P004", "P005", "P006"],
    "product_name":  ["Laptop", "Mouse", "Keyboard", "Monitor", "Headset", "Webcam"],
    "category":      ["Electronics", "Accessories", "Accessories", "Electronics", "Accessories", "Electronics"],
    "unit_price":    [1200.00, 25.50, 45.00, 350.00, 75.00, 60.00],
    "supplier":      ["TechCo", "GearUp", "GearUp", "TechCo", "AudioMax", "TechCo"]
})
products


,product_id,product_name,category,unit_price,supplier
0,P001,Laptop,Electronics,1200.0,TechCo
1,P002,Mouse,Accessories,25.5,GearUp
2,P003,Keyboard,Accessories,45.0,GearUp
3,P004,Monitor,Electronics,350.0,TechCo
4,P005,Headset,Accessories,75.0,AudioMax
5,P006,Webcam,Electronics,60.0,TechCo


### 1.3 Orders fact table


In [4]:
np.random.seed(42)

n_orders = 30
orders = pd.DataFrame({
    "order_id":      range(1, n_orders + 1),
    "customer_id":   np.random.choice([101, 102, 103, 104, 105, 106, 107, 108, 109, 110], n_orders),
    "product_id":    np.random.choice(["P001", "P002", "P003", "P004", "P005", "P006"], n_orders),
    "quantity":      np.random.randint(1, 6, n_orders),
    "order_date":    pd.date_range("2024-01-01", periods=n_orders, freq="3D"),
    "discount_pct":  np.random.choice([0, 5, 10, 15, None], n_orders),
    "status":        np.random.choice(["completed", "pending", "cancelled", "shipped"], n_orders)
})
orders.head(10)


,order_id,customer_id,product_id,quantity,order_date,discount_pct,status
0,1,107,P001,2,2024-01-01,10,pending
1,2,104,P003,4,2024-01-04,0,shipped
2,3,108,P005,5,2024-01-07,10,completed
3,4,105,P003,2,2024-01-10,10,cancelled
4,5,107,P005,2,2024-01-13,0,cancelled
5,6,110,P001,4,2024-01-16,10,shipped
6,7,103,P002,2,2024-01-19,None,cancelled
7,8,107,P004,2,2024-01-22,5,completed
8,9,108,P001,4,2024-01-25,5,shipped
9,10,105,P004,4,2024-01-28,0,completed


## 2. Column Operations

The bread and butter of every transformation layer.


### 2.1 Rename columns


In [5]:
# Single rename
orders_renamed = orders.rename(columns={"discount_pct": "discount_percent"})

# Multiple renames — common when source columns have ugly names
customers_clean = customers.rename(columns={
    "first_name":   "FirstName",
    "last_name":    "LastName",
    "signup_date":  "SignupDate",
    "is_active":    "IsActive",
    "credit_limit": "CreditLimit"
})
customers_clean.columns.tolist()


['customer_id',
 'FirstName',
 'LastName',
 'email',
 'city',
 'country',
 'SignupDate',
 'IsActive',
 'CreditLimit']

### 2.2 Add new columns


In [6]:
# Derived column — full name
customers["full_name"] = customers["first_name"].str.strip() + " " + customers["last_name"]

# Calculated column on the fact table
orders["line_total"] = orders["quantity"] * orders.merge(
    products[["product_id", "unit_price"]], on="product_id", how="left"
)["unit_price"]

orders[["order_id", "product_id", "quantity", "line_total"]].head()


,order_id,product_id,quantity,line_total
0,1,P001,2,2400.0
1,2,P003,4,180.0
2,3,P005,5,375.0
3,4,P003,2,90.0
4,5,P005,2,150.0


### 2.3 Drop columns


In [7]:
# Drop one
orders_slim = orders.drop(columns=["discount_pct"])

# Drop several
customers_slim = customers.drop(columns=["email", "credit_limit"])
customers_slim.columns.tolist()


['customer_id',
 'first_name',
 'last_name',
 'city',
 'country',
 'signup_date',
 'is_active',
 'full_name']

### 2.4 Reorder columns


In [8]:
# Put the most important columns first
col_order = ["order_id", "order_date", "customer_id", "product_id",
             "quantity", "line_total", "status"]
orders_reordered = orders[[c for c in col_order if c in orders.columns]]
orders_reordered.head()


,order_id,order_date,customer_id,product_id,quantity,line_total,status
0,1,2024-01-01,107,P001,2,2400.0,pending
1,2,2024-01-04,104,P003,4,180.0,shipped
2,3,2024-01-07,108,P005,5,375.0,completed
3,4,2024-01-10,105,P003,2,90.0,cancelled
4,5,2024-01-13,107,P005,2,150.0,cancelled


## 3. Data Type Casting

Getting types right is critical — wrong types break joins, date filters and aggregations.


In [9]:
# Check current types
print(customers.dtypes)


customer_id       int64
first_name          str
last_name           str
email               str
city                str
country             str
signup_date         str
is_active         int64
credit_limit    float64
full_name           str
dtype: object


In [10]:
# signup_date came in as a string — cast to datetime
customers["signup_date"] = pd.to_datetime(customers["signup_date"])

# is_active is int but should be boolean for clarity
customers["is_active_bool"] = customers["is_active"].astype(bool)

# credit_limit has NaN so it became float — keep as float or fill + cast
customers["credit_limit_int"] = customers["credit_limit"].fillna(0).astype(int)

print(customers[["signup_date", "is_active", "is_active_bool", "credit_limit", "credit_limit_int"]].dtypes)


signup_date         datetime64[us]
is_active                    int64
is_active_bool                bool
credit_limit               float64
credit_limit_int             int64
dtype: object


In [11]:
# Casting numeric strings
sample = pd.DataFrame({"amount": ["100", "200.5", "300", None, "abc"]})

# Safe casting with errors='coerce' — invalid values become NaN instead of crashing
sample["amount_numeric"] = pd.to_numeric(sample["amount"], errors="coerce")
sample


,amount,amount_numeric
0,100,100.0
1,200.5,200.5
2,300,300.0
3,NaN,NaN
4,abc,NaN


## 4. String Transformations

Source data is messy: extra spaces, inconsistent casing, embedded junk. Clean it.


In [12]:
# ── Case normalization ──
customers["first_name_clean"] = customers["first_name"].str.strip().str.title()
customers["last_name_clean"]  = customers["last_name"].str.strip().str.title()
customers["city_clean"]       = customers["city"].str.strip().str.title()

customers[["first_name", "first_name_clean", "last_name", "last_name_clean", "city", "city_clean"]].head()


,first_name,first_name_clean,last_name,last_name_clean,city,city_clean
0,Alice,Alice,Smith,Smith,Nairobi,Nairobi
1,Bob,Bob,JONES,Jones,London,London
2,Charlie,Charlie,brown,Brown,nairobi,Nairobi
3,Diana,Diana,Williams,Williams,New York,New York
4,Eve,Eve,Davis,Davis,London,London


In [13]:
# ── Upper / Lower ──
customers["country_upper"] = customers["country"].str.upper()
customers["email_lower"]   = customers["email"].str.lower()

# ── Contains (filtering pattern) ──
kenyan_customers = customers[customers["country"].str.contains("Kenya", na=False)]
print(f"Kenyan customers: {len(kenyan_customers)}")


Kenyan customers: 4


In [14]:
# ── Split and extract ──
customers["email_domain"] = customers["email"].str.split("@").str[1]

# ── Replace ──
customers["email_masked"] = customers["email"].str.replace(
    r"^(.{2}).*(@.*)", r"\1***\2", regex=True
)
customers[["email", "email_domain", "email_masked"]].head()


,email,email_domain,email_masked
0,alice@mail.com,mail.com,al***@mail.com
1,bob@mail.com,mail.com,bo***@mail.com
2,charlie@mail.com,mail.com,ch***@mail.com
3,NaN,NaN,NaN
4,eve@mail.com,mail.com,ev***@mail.com


In [ ]:
# ── Padding and slicing ──
products["product_code"] = products["product_id"].str.replace("P", "PROD-").str.zfill(10)

# ── Length ──
customers["name_length"] = customers["full_name"].str.len()

# ── Concatenation with separator ──
customers["location"] = customers["city"].str.cat(customers["country"], sep=", ", na_rep="Unknown")
customers[["full_name", "name_length", "location"]].head()


## 5. Date and Time Transformations

Dates are central to every warehouse — fiscal periods, aging, SCD2 effective dates.


In [15]:
# Make sure we have datetime
orders["order_date"] = pd.to_datetime(orders["order_date"])

# ── Extract components ──
orders["order_year"]    = orders["order_date"].dt.year
orders["order_month"]   = orders["order_date"].dt.month
orders["order_day"]     = orders["order_date"].dt.day
orders["order_weekday"] = orders["order_date"].dt.day_name()
orders["order_quarter"] = orders["order_date"].dt.quarter

orders[["order_id", "order_date", "order_year", "order_month", "order_weekday", "order_quarter"]].head()


,order_id,order_date,order_year,order_month,order_weekday,order_quarter
0,1,2024-01-01,2024,1,Monday,1
1,2,2024-01-04,2024,1,Thursday,1
2,3,2024-01-07,2024,1,Sunday,1
3,4,2024-01-10,2024,1,Wednesday,1
4,5,2024-01-13,2024,1,Saturday,1


In [16]:
# ── Date arithmetic ──
orders["delivery_estimate"] = orders["order_date"] + timedelta(days=7)
orders["days_since_order"]  = (pd.Timestamp.now() - orders["order_date"]).dt.days

# ── Period formatting ──
orders["year_month"] = orders["order_date"].dt.to_period("M").astype(str)

# ── Fiscal year (starting April) ──
orders["fiscal_year"] = orders["order_date"].apply(
    lambda d: d.year if d.month >= 4 else d.year - 1
)

orders[["order_id", "order_date", "delivery_estimate", "days_since_order", "year_month", "fiscal_year"]].head()


,order_id,order_date,delivery_estimate,days_since_order,year_month,fiscal_year
0,1,2024-01-01,2024-01-08,1022,2024-01,2023
1,2,2024-01-04,2024-01-11,1019,2024-01,2023
2,3,2024-01-07,2024-01-14,1016,2024-01,2023
3,4,2024-01-10,2024-01-17,1013,2024-01,2023
4,5,2024-01-13,2024-01-20,1010,2024-01,2023


In [17]:
# ── Floor / Ceil to week or month ──
orders["week_start"] = orders["order_date"].dt.to_period("W").apply(lambda r: r.start_time)
orders["month_start"] = orders["order_date"].dt.to_period("M").apply(lambda r: r.start_time)

orders[["order_date", "week_start", "month_start"]].head()


,order_date,week_start,month_start
0,2024-01-01,2024-01-01,2024-01-01
1,2024-01-04,2024-01-01,2024-01-01
2,2024-01-07,2024-01-01,2024-01-01
3,2024-01-10,2024-01-08,2024-01-01
4,2024-01-13,2024-01-08,2024-01-01


## 6. Handling Nulls and Missing Data

Nulls are the number one source of broken reports. Handle them explicitly.


In [18]:
# ── Check for nulls ──
print("=== Null counts ===")
print(customers.isnull().sum())


=== Null counts ===
customer_id         0
first_name          1
last_name           0
email               1
city                1
country             1
signup_date         0
is_active           0
credit_limit        2
full_name           1
is_active_bool      0
credit_limit_int    0
first_name_clean    1
last_name_clean     0
city_clean          1
country_upper       1
email_lower         1
email_domain        1
email_masked        1
dtype: int64


In [19]:
# ── Fill with a default ──
customers["credit_limit_filled"] = customers["credit_limit"].fillna(0)

# ── Fill with the median (good for numeric columns) ──
median_limit = customers["credit_limit"].median()
customers["credit_limit_median"] = customers["credit_limit"].fillna(median_limit)

# ── Fill string columns with 'Unknown' ──
customers["city_filled"]  = customers["city"].fillna("Unknown")
customers["email_filled"] = customers["email"].fillna("no-email@placeholder.com")

customers[["customer_id", "credit_limit", "credit_limit_filled", "credit_limit_median"]].head()


,customer_id,credit_limit,credit_limit_filled,credit_limit_median
0,101,5000.0,5000.0,5000.0
1,102,7500.0,7500.0,7500.0
2,103,3000.0,3000.0,3000.0
3,104,10000.0,10000.0,10000.0
4,105,6000.0,6000.0,6000.0


In [20]:
# ── Forward fill / Backward fill (useful for time series) ──
ts = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=7),
    "value": [10, None, None, 25, None, 30, None]
})
ts["ffill"] = ts["value"].ffill()   # carry last known value forward
ts["bfill"] = ts["value"].bfill()   # carry next known value backward
ts


,date,value,ffill,bfill
0,2024-01-01,10.0,10.0,10.0
1,2024-01-02,NaN,10.0,25.0
2,2024-01-03,NaN,10.0,25.0
3,2024-01-04,25.0,25.0,25.0
4,2024-01-05,NaN,25.0,30.0
5,2024-01-06,30.0,30.0,30.0
6,2024-01-07,NaN,30.0,NaN


In [21]:
# ── Drop rows with nulls ──
# Drop if ANY column is null
customers_no_nulls = customers.dropna(subset=["email", "city", "credit_limit"])
print(f"Before: {len(customers)} rows → After: {len(customers_no_nulls)} rows")


Before: 10 rows → After: 7 rows


In [22]:
# ── Coalesce pattern (first non-null from multiple columns) ──
customers["contact"] = customers["email"].combine_first(
    customers["first_name"].str.cat(customers["last_name"], sep=" ")
)
customers[["customer_id", "email", "first_name", "contact"]].head()


,customer_id,email,first_name,contact
0,101,alice@mail.com,Alice,alice@mail.com
1,102,bob@mail.com,Bob,bob@mail.com
2,103,charlie@mail.com,Charlie,charlie@mail.com
3,104,NaN,Diana,Diana Williams
4,105,eve@mail.com,Eve,eve@mail.com


## 7. Filtering and Conditional Logic

Select the rows you need, add flag columns, apply business rules.


In [23]:
# ── Simple filter ──
active_customers = customers[customers["is_active"] == 1]
print(f"Active customers: {len(active_customers)}")

# ── Multiple conditions (AND = &, OR = |) ──
kenya_active = customers[
    (customers["country"] == "Kenya") & (customers["is_active"] == 1)
]
print(f"Active Kenyan customers: {len(kenya_active)}")

# ── isin() for matching a list ──
uk_kenya = customers[customers["country"].isin(["Kenya", "UK"])]
print(f"UK or Kenya: {len(uk_kenya)}")


Active customers: 7
Active Kenyan customers: 1
UK or Kenya: 7


In [24]:
# ── Conditional column with np.where (like CASE WHEN in SQL) ──
orders["is_large_order"] = np.where(orders["quantity"] >= 3, "Yes", "No")

# ── Multiple conditions with np.select (like nested CASE WHEN) ──
conditions = [
    orders["status"] == "completed",
    orders["status"] == "shipped",
    orders["status"] == "pending",
    orders["status"] == "cancelled"
]
labels = ["Fulfilled", "In Transit", "Awaiting", "Void"]
orders["status_label"] = np.select(conditions, labels, default="Other")

orders[["order_id", "status", "status_label", "quantity", "is_large_order"]].head(10)


,order_id,status,status_label,quantity,is_large_order
0,1,pending,Awaiting,2,No
1,2,shipped,In Transit,4,Yes
2,3,completed,Fulfilled,5,Yes
3,4,cancelled,Void,2,No
4,5,cancelled,Void,2,No
5,6,shipped,In Transit,4,Yes
6,7,cancelled,Void,2,No
7,8,completed,Fulfilled,2,No
8,9,shipped,In Transit,4,Yes
9,10,completed,Fulfilled,4,Yes


In [25]:
# ── .query() for readable filters ──
high_value = orders.query("quantity >= 3 and status == 'completed'")
print(f"High-value completed orders: {len(high_value)}")

# ── Between (date range) ──
jan_orders = orders[orders["order_date"].between("2024-01-01", "2024-01-31")]
print(f"January 2024 orders: {len(jan_orders)}")


High-value completed orders: 3
January 2024 orders: 11


## 10. Joins

Joins connect your fact and dimension tables — the core of any star schema query.


In [ ]:
# ── Inner join (only matching rows) ──
orders_with_products = orders.merge(
    products[["product_id", "product_name", "category", "unit_price"]],
    on="product_id",
    how="inner"
)
orders_with_products[["order_id", "product_id", "product_name", "category", "quantity"]].head()


In [ ]:
# ── Left join (keep all orders, even if product is missing) ──
orders_full = orders.merge(
    customers[["customer_id", "full_name", "country"]],
    on="customer_id",
    how="left"
).merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left"
)
orders_full[["order_id", "full_name", "country", "product_name", "category", "quantity"]].head()
